# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described with a [Croissant schema](https://github.com/mlcommons/croissant), available at the link below:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`. This will download the schema, parse it, and prepare the dataset for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL to the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (not as a dict/list)
meta = dataset.metadata  # meta is of type mlc.Metadata
print(f"Dataset Name: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Display an overview of all record sets and their fields, with their Croissant `@id` values. The `@id` is the unique reference for every schema entity.

In [ ]:
# List available record sets and their fields (referenced by @id)

def summarize_record_sets(ds):
    for record_set in ds.metadata.record_sets:
        print(f"RecordSet name: {record_set.name}\n  RecordSet @id: {record_set.id}\n  Description: {record_set.description}")
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - Field name: {field.name}\n      Field @id: {field.id}\n      Data type: {field.data_type}\n      Description: {field.description}")
        print()

summarize_record_sets(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame. All variables and references use the Croissant `@id` field.

First, choose the main record set for analysis (for this dataset, we expect there should be a primary one holding protein abundance table data).

**Tip:** Copy `@id` values from the previous cell output.

In [ ]:
# Identify record set(s) in the metadata
record_sets = [record_set.id for record_set in dataset.metadata.record_sets]
print("RecordSet @ids in metadata:", record_sets)

# (Choose main record set - here we take the first one)
record_set_id = record_sets[0]

# Load all records for each record set into DataFrames (by @id)
dataframes = {}
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set: {rs_id}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# List the DataFrame columns (which correspond to field @ids)
print("\nDataFrame columns for main record set:")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We will:
- Select a numeric field by its `@id` ([see available columns printed above]).
- Filter records above a threshold.
- Normalize the numeric field.
- Optionally group by a categorical field, using the field `@id`.

**Change the field `@id`s in the code as needed based on your exploration above.**

In [ ]:
# Replace the following with relevant field @ids (as printed above)
# Example: numeric_field_id = 'cr:abundance' (or similar)

# For this example, let's auto-detect a numeric field, or fall back to user input
import numpy as np
df = dataframes[record_set_id]

# Try to find a numeric field in the DataFrame
numeric_field_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Automatically selected numeric field: {numeric_field_id}")
else:
    # Fallback: ask user to input
    numeric_field_id = df.columns[0]  # Just pick first column if none detected
    print(f"No numeric fields detected; using: {numeric_field_id}")

threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

# Filter: take values above the mean (as example threshold)
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows")
print(filtered_df.head())

# Normalize the field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nFirst 5 values of normalized '{numeric_field_id}':")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Group by a categorical field if available (other than the numeric field itself!)
group_fields = [col for col in df.columns if (df[col].dtype == object) and (col != numeric_field_id)]
if group_fields:
    group_field = group_fields[0]
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(grouped_df.head())
else:
    print("\nNo categorical group fields found.")

## 5. Visualization

Visualize the distribution of numeric fields and relationships in the dataset. Below is an example using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (for filtered_df)
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of '{numeric_field_id}' (filtered)")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If a grouped field is available, plot mean values by group
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    top_n = min(20, len(grouped_df))
    sns.barplot(x=grouped_df[group_field].head(top_n), y=grouped_df[numeric_field_id].head(top_n))
    plt.xticks(rotation=90)
    plt.title(f"Mean {numeric_field_id} by {group_field} (top {top_n})")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a FAIR<sup>2</sup> Croissant dataset using the `mlcroissant` library. We:
- Loaded all metadata and records strictly by Croissant `@id` values
- Identified the structure and available fields for each record set
- Performed basic data filtering, normalization, and grouping
- Visualized numeric field distributions and means by group

This workflow is extensible for deeper domain-specific analysis and supports reproducibility by using open metadata standards.